# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# YOUR CODE HERE: đọc data/splits/train.jsonl, xem vài mẫu, vẽ box lên ảnh
# 1a. Thống kê split + kiểm tra rò rỉ phiên quay
import os, sys, json, statistics
from collections import Counter
from pathlib import Path
sys.path.insert(0, os.getcwd())
from scripts.labcommon import CLASSES, read_jsonl

DATA = Path('data')
SPLITS = ('train', 'val', 'test')
rows = {s: read_jsonl(DATA / 'splits' / f'{s}.jsonl') for s in SPLITS}
meta = json.loads((DATA / 'splits' / 'meta.json').read_text(encoding='utf-8'))
print('dataset_version:', meta['dataset_version'])
print('mẫu 1 dòng train:', json.dumps(rows['train'][0], ensure_ascii=False)[:300], '\n')

# Số ảnh / phiên / box từng lớp mỗi split
print(f"{'split':6s}{'ảnh':>6s}{'phiên':>7s}{'box':>6s}" + ''.join(f'{c:>9s}' for c in CLASSES) + f"{'ảnh rỗng':>10s}")
for s in SPLITS:
    cnt = Counter(o['label'] for r in rows[s] for o in r['objects'])
    n_sess = len({r['session'] for r in rows[s]})
    empty = sum(not r['objects'] for r in rows[s])
    print(f"{s:6s}{len(rows[s]):6d}{n_sess:7d}{sum(cnt.values()):6d}"
          + ''.join(f'{cnt[c]:9d}' for c in CLASSES) + f'{empty:10d}')
    assert all(cnt[c] == meta['counts'][s]['boxes'][c] for c in CLASSES), s   # khớp meta.json

# Không phiên nào nằm ở hai split, không ảnh nào bị lặp
sess = {s: {r['session'] for r in rows[s]} for s in SPLITS}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    common = sess[a] & sess[b]
    print(f'phiên chung {a:5s}–{b:5s}: {len(common)}')
    assert not common, common
ids = [r['image_id'] for s in SPLITS for r in rows[s]]
assert len(ids) == len(set(ids)), 'ảnh bị lặp giữa các split'

# Kích thước ảnh và kích thước box (diện tích tương đối so với khung hình)
print('\nkích thước ảnh:', dict(Counter((r['width'], r['height']) for s in SPLITS for r in rows[s])))
all_areas = []
for c in CLASSES:
    areas = [o['yolo'][2] * o['yolo'][3] for s in SPLITS for r in rows[s] for o in r['objects'] if o['label'] == c]
    all_areas += areas
    print(f'{c:8s} n={len(areas):5d}  diện tích trung vị = {100 * statistics.median(areas):.2f}% khung hình')
print(f'{"tất cả":8s} n={len(all_areas):5d}  diện tích trung vị = {100 * statistics.median(all_areas):.2f}% khung hình')

In [ ]:
# 1b. Vẽ 6 ảnh (2 ảnh/split) kèm box gốc từ nhãn YOLO → pixel
import random
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

COLORS = {'pothole': 'red', 'crack': 'yellow', 'manhole': 'cyan'}

def draw_gt(r):
    im = Image.open(DATA / r['image_path']).convert('RGB')
    d = ImageDraw.Draw(im)
    W, H = r['width'], r['height']
    for o in r['objects']:
        xc, yc, w, h = o['yolo']                       # chuẩn hoá [0,1], tâm + kích thước
        box = [(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H]
        d.rectangle(box, outline=COLORS[o['label']], width=3)
        d.text((box[0] + 3, box[1] + 2), o['label'], fill=COLORS[o['label']])
    return im

rng = random.Random(0)
picks = []
for s in SPLITS:
    multi = [r for r in rows[s] if len({o['label'] for o in r['objects']}) >= 2]   # ưu tiên ảnh có ≥ 2 lớp
    pool = multi if len(multi) >= 2 else rows[s]
    picks += [(s, r) for r in rng.sample(pool, 2)]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
for ax, (s, r) in zip(axes.flat, picks):
    ax.imshow(draw_gt(r)); ax.axis('off')
    ax.set_title(f"[{s}] {r['image_id']} · {r['session']} · {len(r['objects'])} box", fontsize=9)
fig.legend(handles=[plt.Line2D([], [], color=c, lw=3, label=l) for l, c in COLORS.items()],
           loc='lower center', ncol=3)
plt.tight_layout()
Path('report/figures').mkdir(parents=True, exist_ok=True)
fig.savefig('report/figures/cp1_gt_samples.jpg', dpi=80, bbox_inches='tight')
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# YOUR CODE HERE: viết yolo_to_box() và objects_to_answer() (GUIDE bước 2)
# 2a. Nhãn YOLO → JSON thang 0–1000 + prompt
import json

def yolo_to_box(label, yolo, coord='norm1000', width=640, height=360):
    """YOLO (xc, yc, w, h chuẩn hoá [0,1]) → {"label", "bbox_2d": [x1, y1, x2, y2]} số nguyên.
    coord='norm1000': thang 0–1000 (quy ước grounding của Qwen3-VL); coord='pixel': pixel ảnh (chỉ dùng cho E4)."""
    xc, yc, w, h = yolo
    sx, sy = (1000, 1000) if coord == 'norm1000' else (width, height)
    x1, y1 = (xc - w / 2) * sx, (yc - h / 2) * sy
    x2, y2 = (xc + w / 2) * sx, (yc + h / 2) * sy
    x1, x2 = (min(max(round(v), 0), sx) for v in (x1, x2))
    y1, y2 = (min(max(round(v), 0), sy) for v in (y1, y2))
    # box siêu nhỏ có thể bị làm tròn thành x2 == x1 → score.py coi CẢ câu trả lời là hỏng; nới ra 1 đơn vị
    if x2 <= x1: x1, x2 = (x1, x1 + 1) if x1 < sx else (x1 - 1, x1)
    if y2 <= y1: y1, y2 = (y1, y1 + 1) if y1 < sy else (y1 - 1, y1)
    return {'label': label, 'bbox_2d': [x1, y1, x2, y2]}

def objects_to_answer(objects, coord='norm1000', width=640, height=360):
    boxes = [yolo_to_box(o['label'], o['yolo'], coord, width, height) for o in objects]
    boxes.sort(key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))   # trên→dưới, trái→phải: thứ tự ổn định cho model học
    return json.dumps(boxes, separators=(',', ':'))                 # "[]" nếu không có object

def make_prompt(coord='norm1000', width=640, height=360):
    if coord == 'norm1000':
        scale = ('integers on a 0-1000 scale relative to the image width and height: '
                 '(0,0) is the top-left corner and (1000,1000) is the bottom-right corner, NOT pixels')
    else:
        scale = f'integer pixel coordinates of this {width}x{height} image: (0,0) is the top-left corner'
    return ('Detect all road damage in this dashcam image. Classes: "pothole", "crack", "manhole".\n'
            'Return ONLY a JSON list, with no explanation and no markdown, in the form '
            '[{"label": "pothole", "bbox_2d": [x1, y1, x2, y2]}].\n'
            f'Coordinates are {scale}.\n'
            'If there is no damage, return [].')

PROMPT = make_prompt('norm1000')
print(PROMPT, '\n')
from scripts.labcommon import read_jsonl
_r = read_jsonl('data/splits/train.jsonl')[0]
print('nhãn YOLO :', _r['objects'])
print('đáp án    :', objects_to_answer(_r['objects']))

In [ ]:
# 2b. Kiểm tra: JSON → pixel phải trùng box gốc; trọng tài đọc được nhãn; lệch thang làm F1 sụp
import os, sys
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
sys.path.insert(0, os.getcwd())
from scripts.labcommon import read_jsonl
from scripts.score import parse_output, evaluate

DATA = Path('data')
rows = {s: read_jsonl(DATA / 'splits' / f'{s}.jsonl') for s in ('train', 'val', 'test')}

def gt_pixel(r):      # box gốc YOLO → pixel (kẹp trong ảnh)
    W, H = r['width'], r['height']
    return [(max((xc - w / 2) * W, 0), max((yc - h / 2) * H, 0), min((xc + w / 2) * W, W), min((yc + h / 2) * H, H))
            for xc, yc, w, h in (o['yolo'] for o in r['objects'])]

def answer_to_pixel(answer, W, H):   # JSON 0–1000 → pixel, đọc bằng chính parse_output của trọng tài
    return [(x1 / 1000 * W, y1 / 1000 * H, x2 / 1000 * W, y2 / 1000 * H) for x1, y1, x2, y2 in
            (p['bbox'] for p in parse_output(answer))]

# (1) Round-trip trên TẤT CẢ ảnh: mỗi box gốc phải có box quy ngược lệch < 1 pixel
max_err, n_box = 0.0, 0
for rs in rows.values():
    for r in rs:
        ans = objects_to_answer(r['objects'])
        assert parse_output(ans) is not None, (r['image_id'], ans)       # trọng tài đọc được
        back = answer_to_pixel(ans, r['width'], r['height'])
        assert len(back) == len(r['objects'])
        for g in gt_pixel(r):
            err = min(max(abs(a - b) for a, b in zip(g, p)) for p in back)
            max_err, n_box = max(max_err, err), n_box + 1
print(f'round-trip {n_box} box: sai số lớn nhất = {max_err:.2f} px (do làm tròn về số nguyên 0–1000)')
assert max_err < 1.0

# (2) Vẽ lại: box gốc (xanh lá, dày) và box quy ngược từ JSON (đỏ, mảnh) phải chồng khít
r = max(rows['test'], key=lambda r: len(r['objects']))
im = Image.open(DATA / r['image_path']).convert('RGB'); d = ImageDraw.Draw(im)
for g in gt_pixel(r):
    d.rectangle(g, outline='lime', width=5)
for p in answer_to_pixel(objects_to_answer(r['objects']), r['width'], r['height']):
    d.rectangle(p, outline='red', width=1)
plt.figure(figsize=(12, 7)); plt.imshow(im); plt.axis('off')
plt.title(f"{r['image_id']}: xanh = box gốc YOLO, đỏ = JSON 0–1000 quy ngược về pixel\n"
          f"{objects_to_answer(r['objects'])[:150]}…", fontsize=9)
Path('report/figures').mkdir(parents=True, exist_ok=True)
plt.savefig('report/figures/cp2_roundtrip.jpg', dpi=80, bbox_inches='tight'); plt.show()

# (3) Chấm "đáp án hoàn hảo" bằng trọng tài: đúng thang → F1 = 1; lệch thang → F1 sụp (bằng chứng Câu 3)
print('\nchấm đáp án hoàn hảo trên test bằng scripts/score.py:')
for label_coord, score_coord in [('norm1000', 'norm1000'), ('pixel', 'pixel'),
                                 ('norm1000', 'pixel'), ('pixel', 'norm1000')]:
    preds = {r['image_id']: objects_to_answer(r['objects'], label_coord, r['width'], r['height'])
             for r in rows['test']}
    m = evaluate(rows['test'], preds, coord=score_coord, n_boot=200)
    print(f"  nhãn {label_coord:8s} | chấm --coord {score_coord:8s} → F1 = {m['micro']['f1']:.3f}  "
          f"mean IoU = {m['mean_iou']:.3f}  FP/ảnh = {m['fp_per_image']:.2f}")

# (4) Độ dài đáp án theo token: đáp án dài hơn max_new_tokens (256) sẽ bị cắt → JSON hỏng
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-VL-4B-Instruct')
for s, rs in rows.items():
    lens = [len(tok(objects_to_answer(r['objects'])).input_ids) + 1 for r in rs]   # +1 cho <|im_end|>
    nbox = [len(r['objects']) for r in rs]
    print(f'{s:5s} token đáp án: trung bình {sum(lens) / len(lens):5.1f}, lớn nhất {max(lens):4d} '
          f'(ảnh {max(nbox)} box) | số ảnh > 256 token: {sum(l > 256 for l in lens)}')
print(f'≈ {(lens[nbox.index(max(nbox))] - 2) / max(max(nbox), 1):.0f} token mỗi box')


## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)
# 3a. Nạp Qwen3-VL-4B 4-bit (ViT + lm_head giữ fp16) và suy luận zero-shot trên test
import os, sys, json, time, shutil
from collections import Counter
from pathlib import Path
import torch
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
sys.path.insert(0, os.getcwd())
from scripts.labcommon import read_jsonl

assert 'PROMPT' in globals(), 'Chạy ô 2a trước (định nghĩa PROMPT, objects_to_answer)'
MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
DATA = Path('data')
DRIVE_DIR = Path('/content/drive/MyDrive/lab10_2A202602818')   # sao lưu: session Colab có thể bị ngắt

def load_base_model():
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                               bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
                               # ViT (cả projector) + lm_head giữ fp16. transformers 5.x so khớp tên module từ ĐẦU chuỗi
                               # (re.match) hoặc theo đuôi, nên phải ghi 'model.visual': chỉ 'visual' sẽ KHÔNG khớp
                               # 'model.visual.blocks...' và ViT bị lượng tử hoá (đã gặp: 104 Linear4bit trong visual).
                               llm_int8_skip_modules=['model.visual', 'lm_head'])
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=quant, dtype=torch.float16, device_map={'': 0})
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    processor.tokenizer.padding_side = 'left'                               # batch generate: pad bên trái
    return model, processor

def vit_is_fp16(m):
    return not any('4bit' in type(x).__name__ for n, x in m.named_modules() if '.visual.' in f'.{n}.')

if 'model' in globals() and not vit_is_fp16(model):   # model cũ trong bộ nhớ bị lượng tử hoá sai → bỏ, nạp lại
    import gc; del model; globals().pop('m', None); gc.collect(); torch.cuda.empty_cache()
if 'model' not in globals():          # chạy lại ô thì không nạp lại model (≈ 2–3 phút)
    t0 = time.time()
    model, processor = load_base_model()
    print(f'nạp model: {time.time() - t0:.0f}s | VRAM {torch.cuda.memory_allocated() / 1e9:.2f} GB')
model.eval()

# Xác nhận: Linear của ViT/projector và lm_head là fp16 thường, của LLM là 4-bit
def linear_kinds(model):   # trong hàm: biến vòng lặp không giữ module của model cũ trong bộ nhớ
    kinds = Counter()
    for n, m in model.named_modules():
        if isinstance(m, torch.nn.Linear):
            part = 'visual' if '.visual.' in f'.{n}.' else 'lm_head' if n.endswith('lm_head') else 'LLM'
            kinds[(part, type(m).__name__, str(m.weight.dtype))] += 1
    return kinds
kinds = linear_kinds(model)
for k, v in sorted(kinds.items()):
    print(f'  {k[0]:8s} {k[1]:10s} {k[2]:15s} × {v}')
assert all(t == 'Linear' for (part, t, _) in kinds if part == 'visual'), 'ViT bị lượng tử hoá!'

def generate_batch(images, prompt, max_new_tokens=256):
    msgs = [[{'role': 'user', 'content': [{'type': 'image', 'image': im},
                                          {'type': 'text', 'text': prompt}]}] for im in images]
    inputs = processor.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_dict=True,
                                           return_tensors='pt', padding=True).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def predict(rows, out_path, prompt, batch_size=4, max_new_tokens=256):
    """Ghi {"image_id", "raw"} từng dòng. Chạy lại thì bỏ qua ảnh đã có (khôi phục từ Drive nếu mất work/)."""
    out_path = Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
    backup = DRIVE_DIR / out_path.name
    if not out_path.exists() and backup.exists():
        shutil.copy(backup, out_path); print('khôi phục từ Drive:', backup)
    done = {r['image_id'] for r in read_jsonl(out_path)} if out_path.exists() else set()
    todo = [r for r in rows if r['image_id'] not in done]
    print(f'{out_path}: đã có {len(done)}, còn {len(todo)} ảnh')
    t0 = time.time()
    with open(out_path, 'a', encoding='utf-8') as f:
        for i in range(0, len(todo), batch_size):
            batch = todo[i:i + batch_size]
            images = [Image.open(DATA / r['image_path']).convert('RGB') for r in batch]
            for r, raw in zip(batch, generate_batch(images, prompt, max_new_tokens)):
                f.write(json.dumps({'image_id': r['image_id'], 'raw': raw}, ensure_ascii=False) + '\n')
            f.flush()
            n = i + len(batch)
            if (i // batch_size) % 10 == 9 or n == len(todo):
                el = time.time() - t0
                print(f'  {n}/{len(todo)} ảnh | {el / 60:.1f} phút | còn ≈ {el / n * (len(todo) - n) / 60:.1f} phút')
                if DRIVE_DIR.parent.exists():
                    DRIVE_DIR.mkdir(parents=True, exist_ok=True); shutil.copy(out_path, backup)
    return (time.time() - t0) / 60

# Thử trước trên 2 ảnh val: số token ảnh (640×360 phải là 220) và output thô
val = read_jsonl(DATA / 'splits' / 'val.jsonl')
im = Image.open(DATA / val[0]['image_path']).convert('RGB')
enc = processor.apply_chat_template([{'role': 'user', 'content': [{'type': 'image', 'image': im},
                                     {'type': 'text', 'text': PROMPT}]}],
                                    tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt')
img_pad = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
print('\nsố token ảnh:', int((enc['input_ids'] == img_pad).sum()), '| tổng token prompt:', enc['input_ids'].shape[1])
for r, raw in zip(val[:2], generate_batch([Image.open(DATA / r['image_path']).convert('RGB') for r in val[:2]], PROMPT)):
    print(f"\n[{r['image_id']}] GT : {objects_to_answer(r['objects'])}\n  raw: {raw!r}")

# Chạy E0 trên toàn bộ test
test = read_jsonl(DATA / 'splits' / 'test.jsonl')
minutes = predict(test, 'work/e0.jsonl', PROMPT)
print(f'\nE0 xong: {minutes:.1f} phút | VRAM đỉnh {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

In [ ]:
# 3b. Phân tích lỗi E0 (Câu 2): loại lỗi, thang toạ độ, 5 output thô
from collections import Counter
from scripts.score import parse_output, evaluate
from scripts.labcommon import CLASSES, read_jsonl

test = read_jsonl('data/splits/test.jsonl')
preds = {r['image_id']: r['raw'] for r in read_jsonl('work/e0.jsonl')}
ev = json.load(open('report/evidence/e0.json', encoding='utf-8'))
print(f"parse_valid_rate = {ev['parse_valid_rate']:.3f} | F1 = {ev['micro']['f1']:.3f} "
      f"CI95 = {ev['f1_ci95']} | FP/ảnh = {ev['fp_per_image']:.2f} | mean IoU = {ev['mean_iou']:.3f}")

kinds, labels, n_pred, n_gt, coords = Counter(), Counter(), 0, 0, []
for e in test:
    raw = preds[e['image_id']]
    p = parse_output(raw)
    n_gt += len(e['objects'])
    if '```' in raw: kinds['bọc ```json (score.py vẫn đọc được)'] += 1
    if p is None:
        kinds['JSON hỏng'] += 1
        if raw.count('[') > raw.count(']') or raw.count('{') > raw.count('}'):
            kinds['  ↳ bị cắt giữa chừng (chưa đóng ] hoặc })'] += 1
        continue
    n_pred += len(p)
    labels.update(q['label'] for q in p)
    coords += [q['bbox'] for q in p]
    if not p and e['objects']: kinds['trả [] dù ảnh có hư hại'] += 1
    if any(q['label'] not in CLASSES for q in p): kinds['có nhãn ngoài 3 lớp'] += 1
    if len(p) > len(e['objects']): kinds['nhiều box hơn GT'] += 1
    if len(p) < len(e['objects']): kinds['ít box hơn GT'] += 1
    if len({q['bbox'] for q in p}) < len(p): kinds['lặp lại cùng một box'] += 1
print(f'\n{len(test)} ảnh test | box dự đoán {n_pred} vs box thật {n_gt}')
for k, v in kinds.most_common():
    print(f'  {k:45s} {v:4d} ảnh')
print('nhãn dự đoán:', dict(labels.most_common()))

# Thang toạ độ: nếu model trả pixel (x ≤ 640, y ≤ 360) thì chấm --coord pixel sẽ cho F1 cao hơn
if coords:
    xs = [v for b in coords for v in (b[0], b[2])]; ys = [v for b in coords for v in (b[1], b[3])]
    print(f'toạ độ dự đoán: x ∈ [{min(xs):.0f}, {max(xs):.0f}], y ∈ [{min(ys):.0f}, {max(ys):.0f}]')
for coord in ('norm1000', 'pixel'):
    m = evaluate(test, preds, coord=coord, n_boot=200)
    print(f"  chấm E0 với --coord {coord:8s}: F1 = {m['micro']['f1']:.3f}  mean IoU = {m['mean_iou']:.3f}")

print('\n5 output thô đầu tiên:')
for e in test[:5]:
    print(f"\n[{e['image_id']}] GT : {objects_to_answer(e['objects'])}\n  raw: {preds[e['image_id']][:400]!r}")

# Sao lưu bằng chứng sang Drive (và tải về máy để đưa vào repo)
if DRIVE_DIR.parent.exists():
    shutil.copy('report/evidence/e0.json', DRIVE_DIR / 'e0.json'); print('\nđã sao lưu e0.json sang', DRIVE_DIR)

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)
# 4a. Ví dụ huấn luyện: chat template + mask loss (chỉ học phần trả lời) + collate + Dataset
import os, sys, random
from pathlib import Path
import torch
from PIL import Image
from transformers import AutoProcessor
sys.path.insert(0, os.getcwd())
from scripts.labcommon import read_jsonl

assert 'PROMPT' in globals(), 'Chạy ô 2a trước (PROMPT, objects_to_answer)'
MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
DATA = Path('data')
if 'processor' not in globals():      # chỉ cần processor, không cần nạp model
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    processor.tokenizer.padding_side = 'left'
IM_END = processor.tokenizer.convert_tokens_to_ids('<|im_end|>')
IMG_PAD = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')

def encode_example(image, prompt, answer):
    user = {'role': 'user', 'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': prompt}]}
    asst = {'role': 'assistant', 'content': [{'type': 'text', 'text': answer}]}
    kw = dict(tokenize=True, return_dict=True, return_tensors='pt')
    prompt_enc = processor.apply_chat_template([user], add_generation_prompt=True, **kw)
    full_enc = processor.apply_chat_template([user, asst], add_generation_prompt=False, **kw)   # không cắt ngắn
    n = prompt_enc['input_ids'].shape[1]
    ids = full_enc['input_ids'][0]
    assert torch.equal(ids[:n], prompt_enc['input_ids'][0]), 'prompt không phải tiền tố của chuỗi đầy đủ'
    labels = ids.clone()
    labels[:n] = -100                                     # prompt + 220 token ảnh: không tính loss
    end = n + int((ids[n:] == IM_END).nonzero()[0])       # <|im_end|> kết thúc câu trả lời
    labels[end + 1:] = -100                               # '\n' sau <|im_end|>: model dừng ở <|im_end|>, không cần học
    out = {'input_ids': ids, 'attention_mask': full_enc['attention_mask'][0], 'labels': labels,
           'pixel_values': full_enc['pixel_values'], 'image_grid_thw': full_enc['image_grid_thw']}
    if 'mm_token_type_ids' in full_enc:
        out['mm_token_type_ids'] = full_enc['mm_token_type_ids'][0]
    return out

def collate(batch):
    L = max(len(b['input_ids']) for b in batch)
    def pad(key, value):                                  # pad bên phải đến cùng độ dài
        return torch.stack([torch.cat([b[key], torch.full((L - len(b[key]),), value, dtype=b[key].dtype)])
                            for b in batch])
    out = {'input_ids': pad('input_ids', processor.tokenizer.pad_token_id),
           'attention_mask': pad('attention_mask', 0),
           'labels': pad('labels', -100)}                 # token đệm không tính loss
    if 'mm_token_type_ids' in batch[0]:
        out['mm_token_type_ids'] = pad('mm_token_type_ids', 0)
    out['pixel_values'] = torch.cat([b['pixel_values'] for b in batch])       # nối theo chiều 0, KHÔNG pad
    out['image_grid_thw'] = torch.cat([b['image_grid_thw'] for b in batch])
    return out

class GroundingDataset(torch.utils.data.Dataset):
    """fraction < 1: lấy ngẫu nhiên (cố định theo seed) một phần ảnh train, dùng cho E4 data_size."""
    def __init__(self, rows, prompt, coord='norm1000', fraction=1.0, seed=42):
        rows = list(rows)
        if fraction < 1.0:
            rows = random.Random(seed).sample(rows, max(1, round(len(rows) * fraction)))
        self.rows, self.prompt, self.coord = rows, prompt, coord
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        image = Image.open(DATA / r['image_path']).convert('RGB')
        answer = objects_to_answer(r['objects'], self.coord, r['width'], r['height'])
        return encode_example(image, self.prompt, answer)

print('đã định nghĩa encode_example, collate, GroundingDataset')

In [ ]:
# 4b. Kiểm tra: 220 token ảnh; phần có loss giải mã ra đúng JSON + <|im_end|>; collate đúng
tok = processor.tokenizer
train_rows = read_jsonl(DATA / 'splits' / 'train.jsonl')
ds = GroundingDataset(train_rows, PROMPT)

r, ex = train_rows[0], ds[0]
answer = objects_to_answer(r['objects'])
learn = ex['labels'] != -100
n_img = int((ex['input_ids'] == IMG_PAD).sum())
decoded = tok.decode(ex['input_ids'][learn])
print('chuỗi đầy đủ (gộp 220 token ảnh):')
print(tok.decode(ex['input_ids'][ex['input_ids'] != IMG_PAD]).replace('<|vision_start|>', '<|vision_start|>[220 × <|image_pad|>]'))
print(f'\nsố token ảnh: {n_img}')
print(f'phần có loss (labels != -100): {decoded!r}')
assert n_img == 220, n_img
assert decoded == answer + '<|im_end|>', decoded

L, n_learn = len(ex['input_ids']), int(learn.sum())
first = int(learn.nonzero()[0])
print(f'\nví dụ 0: tổng {L} token = prompt {first} (trong đó ảnh {n_img}, chữ {first - n_img}) '
      f'+ trả lời {n_learn} + {L - first - n_learn} token đuôi')
print(f'→ loss chỉ tính trên {n_learn}/{L} = {100 * n_learn / L:.1f}% token của chuỗi')

# Cả tập train: prompt có độ dài cố định (ảnh cùng 640×360, cùng PROMPT) nên chỉ cần đếm token đáp án
ans_len = [len(tok(objects_to_answer(r['objects'])).input_ids) + 1 for r in train_rows]   # +1: <|im_end|>
seq_len = [first + a + 1 for a in ans_len]
print(f'train {len(train_rows)} ảnh: token trả lời TB {sum(ans_len) / len(ans_len):.0f} (max {max(ans_len)}) | '
      f'độ dài chuỗi TB {sum(seq_len) / len(seq_len):.0f} (max {max(seq_len)}) | '
      f'tỉ lệ token có loss TB {100 * sum(ans_len) / sum(seq_len):.1f}%')

# collate: ghép ví dụ ngắn nhất và dài nhất
i_min, i_max = ans_len.index(min(ans_len)), ans_len.index(max(ans_len))
b = collate([ds[i_min], ds[i_max]])
print('\ncollate 2 ví dụ (ngắn nhất + dài nhất):')
for k, v in b.items():
    print(f'  {k:18s} {tuple(v.shape)} {v.dtype}')
short = len(ds[i_min]['input_ids'])
assert (b['labels'][0, short:] == -100).all() and (b['attention_mask'][0, short:] == 0).all()   # phần đệm
assert b['pixel_values'].shape[0] == int(b['image_grid_thw'].prod(-1).sum())                  # nối, không pad
for j in range(2):
    assert tok.decode(b['input_ids'][j][b['labels'][j] != -100]).endswith('<|im_end|>')
print('OK: phần đệm có labels = -100 và attention_mask = 0; pixel_values khớp image_grid_thw')

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)
# 5a. apply_setup: đóng băng toàn bộ → LoRA (LLM, + ViT ở E3) → mở projector fp32 (E2, E3)
import re, sys, subprocess, importlib, importlib.util, importlib.metadata
import torch
from packaging.version import Version

# Colab cài sẵn torchao 0.10; peft 0.21.2 thấy torchao < 0.16 là báo ImportError khi gắn LoRA.
# Lab không dùng torchao (lượng tử hoá bằng bitsandbytes) → gỡ đi.
if importlib.util.find_spec('torchao') and Version(importlib.metadata.version('torchao')) < Version('0.16'):
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
    importlib.invalidate_caches()
    assert 'torchao' not in sys.modules, 'torchao đã được import → Runtime → Restart session rồi chạy lại'
    print('đã gỡ torchao bản cũ (không dùng trong lab)')

from peft import LoraConfig, get_peft_model
from peft.tuners.lora import LoraLayer

# Tên module lấy từ cây model thật (model.named_modules()). ViT phải giới hạn trong blocks.N:
# merger và deepstack_merger_list cũng có lớp tên linear_fc1/linear_fc2, không được gắn LoRA vào đó.
LLM_RE = r'model\.language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)'   # 36 × 7 = 252
VIT_RE = r'model\.visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.linear_fc(1|2))'                        # 24 × 4 = 96
PROJECTOR_KEYS = ('visual.merger.', 'visual.deepstack_merger_list.')                                   # ≈ 27M + 82M
SETUPS = {'E1': dict(vit_lora=False, train_projector=False),
          'E2': dict(vit_lora=False, train_projector=True),
          'E3': dict(vit_lora=True,  train_projector=True)}

def apply_setup(model, setup, r=16, alpha=32, dropout=0.05):
    cfg = SETUPS[setup]
    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()                      # cần cho gradient checkpointing khi embedding bị đóng băng
    target = f"({LLM_RE}{'|' + VIT_RE if cfg['vit_lora'] else ''})"
    matched = [n for n, _ in model.named_modules() if re.fullmatch(target, n)]
    peft_model = get_peft_model(model, LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout,
                                                  target_modules=target))   # chuỗi → peft dùng re.fullmatch
    for n, p in peft_model.named_parameters():
        if cfg['train_projector'] and any(k in n for k in PROJECTOR_KEYS):
            p.requires_grad = True                          # projector: huấn luyện ĐẦY ĐỦ, không LoRA
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()                         # mọi tham số huấn luyện ở fp32 (GradScaler + cập nhật nhỏ)
    n_lora = sum(isinstance(m, LoraLayer) for m in peft_model.modules())
    assert n_lora == len(matched), (n_lora, len(matched))
    return peft_model, matched

def trainable_breakdown(model):
    groups = {'LoRA LLM': 0, 'LoRA ViT': 0, 'projector': 0, 'khác': 0}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if 'lora_' in n:
            groups['LoRA ViT' if '.visual.' in n else 'LoRA LLM'] += p.numel()
        elif any(k in n for k in PROJECTOR_KEYS):
            groups['projector'] += p.numel()
        else:
            groups['khác'] += p.numel()
    return groups

def projector_state(model):
    """Tham số projector đã huấn luyện (E2/E3). peft.save_pretrained KHÔNG lưu phần này → phải lưu riêng."""
    return {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad and 'lora_' not in n}

print('đã định nghĩa apply_setup, trainable_breakdown, projector_state')

In [ ]:
# 5b. Kiểm tra trên "model rỗng" (meta device: đủ cấu trúc, không có trọng số, không tốn VRAM)
from transformers import AutoConfig, Qwen3VLForConditionalGeneration
MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
config = AutoConfig.from_pretrained(MODEL_ID)
def meta_model():
    with torch.device('meta'):
        return Qwen3VLForConditionalGeneration._from_config(config, dtype=torch.float16)

# (1) Số tham số từng khối (Câu 1). named_parameters() không đếm trùng lm_head (dùng chung với embed_tokens)
m = meta_model()
blocks = {'ViT (patch_embed + pos_embed + 24 blocks)': 0, 'projector: visual.merger': 0,
          'projector: visual.deepstack_merger_list': 0, 'LLM: embed_tokens (= lm_head)': 0, 'LLM: 36 layers + norm': 0}
for n, p in m.named_parameters():
    if 'visual.merger.' in n: blocks['projector: visual.merger'] += p.numel()
    elif 'visual.deepstack_merger_list.' in n: blocks['projector: visual.deepstack_merger_list'] += p.numel()
    elif '.visual.' in f'.{n}': blocks['ViT (patch_embed + pos_embed + 24 blocks)'] += p.numel()
    elif 'embed_tokens' in n: blocks['LLM: embed_tokens (= lm_head)'] += p.numel()
    else: blocks['LLM: 36 layers + norm'] += p.numel()
TOTAL = sum(blocks.values())
print(f"lm_head dùng chung trọng số với embed_tokens: {m.lm_head.weight is m.model.language_model.embed_tokens.weight}")
for k, v in blocks.items():
    print(f'  {k:42s} {v / 1e6:9.1f}M  ({100 * v / TOTAL:5.2f}%)')
print(f"  {'TỔNG':42s} {TOTAL / 1e6:9.1f}M")
del m

# (2) apply_setup cho từng setup: số module LoRA và số tham số huấn luyện
SETUP_STATS = {}
for setup in ('E1', 'E2', 'E3'):
    pm, matched = apply_setup(meta_model(), setup)
    g = trainable_breakdown(pm)
    n_tr = sum(g.values())
    fp32 = all(p.dtype == torch.float32 for p in pm.parameters() if p.requires_grad)
    SETUP_STATS[setup] = dict(modules=len(matched), trainable=n_tr, **g)
    print(f"\n{setup}: {len(matched)} module LoRA | huấn luyện {n_tr / 1e6:.2f}M = {100 * n_tr / TOTAL:.2f}% "
          f"tổng | mọi tham số huấn luyện fp32: {fp32}")
    print('   ' + ' | '.join(f'{k} {v / 1e6:.2f}M' for k, v in g.items() if v))
    assert fp32 and g['khác'] == 0
    del pm
assert [SETUP_STATS[s]['modules'] for s in ('E1', 'E2', 'E3')] == [252, 252, 348]
d = SETUP_STATS['E2']['trainable'] - SETUP_STATS['E1']['trainable']
print(f'\nE2 − E1 = {d / 1e6:.1f}M tham số (projector, GUIDE: ≈ 109M)')
assert 100e6 < d < 120e6

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)
# 6a. Hàm huấn luyện + nạp lại model đã huấn luyện. Thiết kế để chạy qua đêm:
#   - checkpoint (LoRA + projector + optimizer + scheduler + scaler + bước) lên Drive mỗi 100 bước;
#     bị ngắt thì chạy lại ô 2a, 4a, 5a, 6a, 6c → tự học tiếp từ checkpoint gần nhất, run đã xong thì bỏ qua;
#   - thứ tự dữ liệu cố định theo (seed, bước) nên học tiếp vẫn đúng dữ liệu;
#   - hết VRAM → tự chuyển batch 1 × tích luỹ 8 (cùng batch hiệu dụng 8) và làm lại bước đó.
import os, sys, gc, json, math, time, shutil
from pathlib import Path
import torch, yaml
from transformers import Qwen3VLForConditionalGeneration, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import PeftModel
from peft.tuners.lora import LoraLayer
sys.path.insert(0, os.getcwd())
from scripts.labcommon import read_jsonl

for need in ('make_prompt', 'objects_to_answer', 'GroundingDataset', 'collate', 'apply_setup', 'processor'):
    assert need in globals(), f'Chạy ô 2a, 4a, 5a trước (thiếu {need})'

MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
DATA = Path('data')
USE_CUDA = torch.cuda.is_available()
DRIVE_DIR = Path('/content/drive/MyDrive/lab10_2A202602818')
if USE_CUDA and not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')                 # bắt buộc: checkpoint phải nằm trên Drive mới sống sót khi mất máy ảo
RUNS_DIR = DRIVE_DIR / 'runs' if Path('/content/drive/MyDrive').exists() else Path('work/runs')
RUNS_DIR.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(RUNS_DIR).free / 1e9
print(f'thư mục lưu run: {RUNS_DIR} | còn trống ≈ {free_gb:.1f} GB (mỗi checkpoint E2/E3 ≈ 1,7 GB)')
if free_gb < 4:
    print('CẢNH BÁO: Drive sắp đầy → checkpoint có thể lỗi. Dọn bớt Drive trước khi treo máy.')

def load_base_4bit():
    """Giống E0: LLM NF4 4-bit; ViT + projector + lm_head fp16 ('model.visual' — xem ô 3a)."""
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                               bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
                               llm_int8_skip_modules=['model.visual', 'lm_head'])
    m = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=quant, dtype=torch.float16, device_map={'': 0})
    assert not any('4bit' in type(x).__name__ for n, x in m.named_modules() if '.visual.' in f'.{n}.'), 'ViT bị lượng tử hoá'
    return m

def free_gpu():
    for name in ('model', 'pm', 'eval_model'):           # model của các ô trước (vd. ô 3a)
        globals().pop(name, None)
    gc.collect()
    if USE_CUDA:
        torch.cuda.empty_cache()

def load_cfg(name):
    return yaml.safe_load(open(f'configs/{name}.yaml', encoding='utf-8'))

def atomic_save(obj, path):                               # ghi file tạm rồi đổi tên: mất máy giữa chừng không hỏng file cũ
    tmp = Path(f'{path}.tmp')
    torch.save(obj, tmp)
    os.replace(tmp, path)

def write_json(obj, path):
    tmp = Path(f'{path}.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding='utf-8')
    os.replace(tmp, path)

def strip_peft_prefix(name):
    return name[len('base_model.model.'):] if name.startswith('base_model.model.') else name

def train_run(run_name, setup, r=16, alpha=32, dropout=0.05, lr=2e-4, batch_size=2, grad_accum=4,
              max_steps=300, seed=42, fraction=1.0, coord='norm1000', save_every=100, log_every=10):
    out = RUNS_DIR / run_name
    out.mkdir(parents=True, exist_ok=True)
    if (out / 'run.json').exists():
        info = json.loads((out / 'run.json').read_text(encoding='utf-8'))
        if info.get('done'):
            print(f'[{run_name}] đã huấn luyện xong trước đó → bỏ qua'); return info
    logf = open(out / 'train_log.txt', 'a', encoding='utf-8')
    def say(msg):
        msg = f'[{run_name}] {msg}'
        print(msg, flush=True); logf.write(time.strftime('%H:%M:%S ') + msg + '\n'); logf.flush()

    free_gpu()
    torch.manual_seed(seed)
    model = load_base_4bit()
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model, matched = apply_setup(model, setup, r=r, alpha=alpha, dropout=dropout)
    model.train()                                          # bật dropout LoRA + gradient checkpointing
    device = next(model.parameters()).device
    params = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in params)
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=math.ceil(0.05 * max_steps),
                                            num_training_steps=max_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_CUDA)
    st = dict(step=0, minutes=0.0, peak_vram_gb=0.0, losses=[], nonfinite=0,
              batch_size=batch_size, grad_accum=grad_accum, oom_fallback=False)
    ckpt = out / 'ckpt.pt'
    if ckpt.exists():
        c = torch.load(ckpt, map_location='cpu')
        named = dict(model.named_parameters())
        for n, t in c['trainable'].items():
            named[n].data.copy_(t)
        opt.load_state_dict(c['opt']); sched.load_state_dict(c['sched']); scaler.load_state_dict(c['scaler'])
        st.update(c['state'])
        say(f"học tiếp từ checkpoint bước {st['step']} ({st['minutes']:.1f} phút đã chạy)")
    say(f'{setup}: {len(matched)} module LoRA (r={r}), {n_trainable / 1e6:.2f}M tham số huấn luyện, '
        f'lr={lr}, batch {st["batch_size"]}×{st["grad_accum"]}, {max_steps} bước')

    ds = GroundingDataset(read_jsonl(DATA / 'splits' / 'train.jsonl'), make_prompt(coord), coord=coord,
                          fraction=fraction, seed=seed)
    eff = batch_size * grad_accum                          # batch hiệu dụng cố định = 8 ảnh/bước
    g = torch.Generator().manual_seed(seed)
    order = torch.cat([torch.randperm(len(ds), generator=g)
                       for _ in range(math.ceil(max_steps * eff / len(ds)) + 1)]).tolist()

    def save(final):
        model.save_pretrained(out / 'adapter')             # peft: CHỈ lưu LoRA
        proj = {strip_peft_prefix(n): p.detach().cpu() for n, p in model.named_parameters()
                if p.requires_grad and 'lora_' not in n}
        if proj:
            atomic_save(proj, out / 'projector.pt')        # projector (E2/E3) KHÔNG nằm trong adapter → lưu riêng
        info = dict(run=run_name, setup=setup, r=r, alpha=alpha, dropout=dropout, lr=lr, max_steps=max_steps,
                    seed=seed, fraction=fraction, coord=coord, n_train_images=len(ds), lora_modules=len(matched),
                    trainable_params=n_trainable, peak_vram_gb=round(st['peak_vram_gb'], 2),
                    train_minutes=round(st['minutes'], 2), steps=st['step'], losses=st['losses'],
                    nonfinite_steps=st['nonfinite'], oom_fallback=st['oom_fallback'],
                    batch_size=st['batch_size'], grad_accum=st['grad_accum'], done=final)
        if final:
            write_json(info, out / 'run.json')
            ckpt.unlink(missing_ok=True)                  # xong rồi: bỏ optimizer state cho nhẹ Drive
        else:
            atomic_save(dict(trainable={n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad},
                             opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(),
                             state={k: v for k, v in st.items()}), ckpt)
            write_json(info, out / 'run.json')
        return info

    if USE_CUDA:
        torch.cuda.reset_peak_memory_stats()
    t0, minutes0, s0 = time.time(), st['minutes'], st['step']
    while st['step'] < max_steps:
        s, bs, acc = st['step'], st['batch_size'], st['grad_accum']
        idx = order[s * eff:(s + 1) * eff]
        batch = loss = None
        step_loss, bad = 0.0, False
        try:
            for j in range(acc):
                batch = {k: v.to(device, non_blocking=True)
                         for k, v in collate([ds[i] for i in idx[j * bs:(j + 1) * bs]]).items()}
                with torch.autocast('cuda', dtype=torch.float16, enabled=USE_CUDA):
                    loss = model(**batch).loss / acc
                if not torch.isfinite(loss):
                    bad = True
                    break
                scaler.scale(loss).backward()
                step_loss += loss.item()
        except torch.OutOfMemoryError:
            batch = loss = None
            opt.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache()
            if bs == 1:
                raise
            st.update(batch_size=1, grad_accum=eff, oom_fallback=True)
            say(f'hết VRAM ở bước {s} → chuyển sang batch 1 × tích luỹ {eff} (cùng batch hiệu dụng), làm lại bước')
            continue
        batch = loss = None
        if bad:                                            # tràn số fp16: bỏ bước này, không cập nhật trọng số
            opt.zero_grad(set_to_none=True)
            st['nonfinite'] += 1
            say(f'loss không hữu hạn ở bước {s} (lần {st["nonfinite"]}) → bỏ qua bước')
            if st['nonfinite'] > 10:
                raise RuntimeError('loss nan/inf quá 10 lần: giảm lr hoặc kiểm tra mask (GUIDE: khi gặp sự cố)')
            sched.step(); st['step'] += 1
            continue
        scaler.unscale_(opt)
        gnorm = float(torch.nn.utils.clip_grad_norm_(params, 1.0))
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        st['step'] += 1
        st['losses'].append(round(step_loss, 4))
        st['minutes'] = minutes0 + (time.time() - t0) / 60
        if USE_CUDA:
            st['peak_vram_gb'] = max(st['peak_vram_gb'], torch.cuda.max_memory_allocated() / 1e9)
        if st['step'] % log_every == 0 or st['step'] in (1, max_steps):
            k = min(log_every, len(st['losses']))
            done_here = st['step'] - s0                    # số bước chạy trong lần này (không tính phần học tiếp)
            eta = (st['minutes'] - minutes0) / max(done_here, 1) * (max_steps - st['step'])
            say(f"bước {st['step']:3d}/{max_steps} | loss {sum(st['losses'][-k:]) / k:.4f} | "
                f"lr {sched.get_last_lr()[0]:.2e} | grad {gnorm:.2f} | {st['minutes']:.1f} phút, còn ≈ {eta:.0f} | "
                f"VRAM đỉnh {st['peak_vram_gb']:.2f} GB")
        if st['step'] % save_every == 0 and st['step'] < max_steps:
            save(final=False); say(f"đã lưu checkpoint bước {st['step']} → {out}")
    info = save(final=True)
    say(f"XONG: {info['train_minutes']:.1f} phút, VRAM đỉnh {info['peak_vram_gb']:.2f} GB, "
        f"loss đầu {sum(st['losses'][:10]) / max(1, len(st['losses'][:10])):.4f} → cuối "
        f"{sum(st['losses'][-10:]) / max(1, len(st['losses'][-10:])):.4f}")
    logf.close()
    del model, opt, sched, scaler, params
    free_gpu()
    return info

def load_trained(run_dir):
    """Nạp model gốc 4-bit → nạp lại projector đã huấn luyện (nếu có) → nạp adapter LoRA. Dùng cho suy luận (CP7)."""
    run_dir = Path(run_dir)
    base = load_base_4bit()
    proj_path = run_dir / 'projector.pt'
    if proj_path.exists():
        sd = torch.load(proj_path, map_location='cpu')
        named = dict(base.named_parameters())
        missing = [k for k in sd if k not in named]
        assert not missing, f'tên tham số projector không khớp: {missing[:3]}'
        k0 = next(iter(sd))
        before = named[k0].detach().float().cpu().clone()
        with torch.no_grad():
            for k, v in sd.items():
                named[k].copy_(v.to(named[k].dtype))      # fp32 đã huấn luyện → fp16 như phần còn lại của ViT
        diff = (named[k0].detach().float().cpu() - before).abs().max().item()
        assert diff > 0, 'projector sau khi nạp giống hệt bản gốc → lưu/nạp sai'
        print(f'  nạp {len(sd)} tensor projector (lệch so với gốc tối đa {diff:.4f})')
    model = PeftModel.from_pretrained(base, run_dir / 'adapter')
    model.eval()
    n_lora = sum(isinstance(m, LoraLayer) for m in model.modules())
    print(f'  nạp adapter: {n_lora} module LoRA')
    return model

print('đã định nghĩa train_run, load_trained')

In [ ]:
# 6b. CHẠY THỬ trước khi treo máy (≈ 5 phút): E3 (setup nặng nhất) 4 bước + lưu + nạp lại + sinh 2 ảnh val.
# Ô này lỗi thì 6c cũng sẽ lỗi → sửa xong mới treo máy. Kết quả chạy thử bị xoá sau khi kiểm tra.
smoke = train_run('_smoke_E3', 'E3', max_steps=4, save_every=2, log_every=1)
print(f"→ ước tính 300 bước: ≈ {smoke['train_minutes'] / 4 * 300:.0f} phút cho E3 | VRAM đỉnh {smoke['peak_vram_gb']:.2f} GB / "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9 if USE_CUDA else 0:.1f} GB")
assert (RUNS_DIR / '_smoke_E3' / 'projector.pt').exists() and (RUNS_DIR / '_smoke_E3' / 'adapter').exists()
eval_model = load_trained(RUNS_DIR / '_smoke_E3')
val2 = read_jsonl(DATA / 'splits' / 'val.jsonl')[:2]
msgs = [[{'role': 'user', 'content': [{'type': 'image', 'image': Image.open(DATA / r['image_path']).convert('RGB')},
                                      {'type': 'text', 'text': PROMPT}]}] for r in val2]
inputs = processor.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_dict=True,
                                       return_tensors='pt', padding=True).to(next(eval_model.parameters()).device)
with torch.inference_mode():
    out = eval_model.generate(**inputs, max_new_tokens=64, do_sample=False)
for r, raw in zip(val2, processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)):
    print(f"  [{r['image_id']}] {raw[:150]!r}")
del eval_model, inputs, out
free_gpu()
shutil.rmtree(RUNS_DIR / '_smoke_E3')
print('\nCHẠY THỬ OK → có thể chạy ô 6c (và xếp hàng ô 7a) rồi đi ngủ')

In [ ]:
# 6c. Huấn luyện E1 → E2 → E3 theo configs/e1–e3.yaml. Bị ngắt: chạy lại 2a, 4a, 5a, 6a, 6c → tự học tiếp.
# Một setup lỗi (vd. hết VRAM cả khi batch 1) không chặn setup sau → qua đêm vẫn có kết quả cho các setup còn lại.
import traceback
RESULTS, ERRORS = {}, {}
for key in ('e1', 'e2', 'e3'):
    cfg = load_cfg(key)
    name = cfg['name']
    assert SETUPS[name]['vit_lora'] == (cfg['lora']['targets'] == 'llm+vit')
    assert SETUPS[name]['train_projector'] == cfg['train_projector']
    try:
        RESULTS[name] = train_run(name, name, r=cfg['lora']['r'], alpha=cfg['lora']['alpha'],
                                  dropout=cfg['lora']['dropout'], lr=float(cfg['lr']), batch_size=cfg['batch_size'],
                                  grad_accum=cfg['grad_accum'], max_steps=cfg['max_steps'], seed=cfg['seed'])
        (RUNS_DIR / name / 'error.txt').unlink(missing_ok=True)
    except Exception:
        ERRORS[name] = traceback.format_exc()
        print(f'!!! {name} LỖI — bỏ qua, chạy setup tiếp theo:\n{ERRORS[name]}', flush=True)
        free_gpu()
for name, err in ERRORS.items():
    (RUNS_DIR / name).mkdir(parents=True, exist_ok=True)
    (RUNS_DIR / name / 'error.txt').write_text(err, encoding='utf-8')
for name, info in RESULTS.items():
    print(f"{name}: {info['trainable_params'] / 1e6:.2f}M tham số | {info['train_minutes']:.1f} phút | "
          f"VRAM đỉnh {info['peak_vram_gb']:.2f} GB | loss {info['losses'][0]:.3f} → {info['losses'][-1]:.3f}")

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)
# 7a. Với mỗi run đã huấn luyện xong: nạp lại (model gốc 4-bit → projector → adapter) → suy luận test → chấm.
# Chạy lại được: ảnh đã suy luận và file bằng chứng đã có thì bỏ qua (sao lưu trên Drive).
import subprocess

def predict_with(model, rows, out_path, prompt, batch_size=4, max_new_tokens=256):
    out_path = Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
    backup = DRIVE_DIR / out_path.name
    if not out_path.exists() and backup.exists():
        shutil.copy(backup, out_path)
    done = {r['image_id'] for r in read_jsonl(out_path)} if out_path.exists() else set()
    todo = [r for r in rows if r['image_id'] not in done]
    print(f'  {out_path}: đã có {len(done)}, còn {len(todo)} ảnh', flush=True)
    device, t0 = next(model.parameters()).device, time.time()
    with open(out_path, 'a', encoding='utf-8') as f:
        for i in range(0, len(todo), batch_size):
            batch = todo[i:i + batch_size]
            msgs = [[{'role': 'user', 'content': [
                {'type': 'image', 'image': Image.open(DATA / r['image_path']).convert('RGB')},
                {'type': 'text', 'text': prompt}]}] for r in batch]
            inputs = processor.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_dict=True,
                                                   return_tensors='pt', padding=True).to(device)
            with torch.inference_mode():
                gen = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            for r, raw in zip(batch, processor.batch_decode(gen[:, inputs['input_ids'].shape[1]:],
                                                            skip_special_tokens=True)):
                f.write(json.dumps({'image_id': r['image_id'], 'raw': raw}, ensure_ascii=False) + '\n')
            f.flush()
            n = i + len(batch)
            if (i // batch_size) % 10 == 9 or n == len(todo):
                el = (time.time() - t0) / 60
                print(f'    {n}/{len(todo)} ảnh | {el:.1f} phút | còn ≈ {el / n * (len(todo) - n):.1f} phút', flush=True)
                if DRIVE_DIR.exists():
                    shutil.copy(out_path, backup)

def evaluate_run(run_name, key, split='test'):
    """key: tên file bằng chứng, vd. 'e1' → work/e1.jsonl, report/evidence/e1.json."""
    info = json.loads((RUNS_DIR / run_name / 'run.json').read_text(encoding='utf-8'))
    assert info['done'], f'{run_name} chưa huấn luyện xong'
    ev = Path(f'report/evidence/{key}.json')
    if not ev.exists() and (DRIVE_DIR / ev.name).exists():
        ev.parent.mkdir(parents=True, exist_ok=True); shutil.copy(DRIVE_DIR / ev.name, ev)
    if ev.exists() and json.loads(ev.read_text(encoding='utf-8')).get('split') == split:
        print(f'[{run_name}] đã có {ev} → bỏ qua'); return json.loads(ev.read_text(encoding='utf-8'))
    print(f'[{run_name}] nạp model đã huấn luyện ...', flush=True)
    free_gpu()
    eval_model = load_trained(RUNS_DIR / run_name)
    rows = read_jsonl(DATA / 'splits' / f'{split}.jsonl')
    pred = Path(f'work/{key}.jsonl')
    predict_with(eval_model, rows, pred, make_prompt(info['coord']))
    del eval_model
    free_gpu()
    cmd = [sys.executable, 'scripts/score.py', '--pred', str(pred), '--name', key.upper(), '--out', str(ev),
           '--split', split, '--coord', info['coord'],
           '--meta', f"trainable_params={info['trainable_params']}",
           '--meta', f"peak_vram_gb={info['peak_vram_gb']:.2f}",
           '--meta', f"train_minutes={info['train_minutes']:.2f}"]
    print(subprocess.run(cmd, check=True, capture_output=True, text=True).stdout)
    if DRIVE_DIR.exists():
        shutil.copy(ev, DRIVE_DIR / ev.name)
    return json.loads(ev.read_text(encoding='utf-8'))

import traceback
EVAL = {}
for name in ('E1', 'E2', 'E3'):
    run_json = RUNS_DIR / name / 'run.json'
    if not (run_json.exists() and json.loads(run_json.read_text(encoding='utf-8')).get('done')):
        print(f'[{name}] chưa huấn luyện xong → bỏ qua'); continue
    try:
        EVAL[name] = evaluate_run(name, name.lower())
    except Exception:
        print(f'!!! {name} LỖI khi suy luận/chấm — bỏ qua:\n{traceback.format_exc()}', flush=True)
        free_gpu()

# Bảng tổng hợp + gói bằng chứng để tải về máy (report/evidence/*.json + run.json + log)
print(f"\n{'':4s} {'F1':>6s} {'CI95':>13s} {'valid':>6s} {'FP/ảnh':>7s} {'pothole':>8s} {'crack':>7s} {'manhole':>8s} {'tham số':>9s} {'VRAM':>6s} {'phút':>6s}")
for key in ('e0', 'e1', 'e2', 'e3'):
    pth = Path(f'report/evidence/{key}.json')
    if not pth.exists():
        continue
    d = json.loads(pth.read_text(encoding='utf-8')); m = d.get('meta', {}); pc = d['per_class']
    print(f"{key.upper():4s} {d['micro']['f1']:6.3f} {d['f1_ci95'][0]:6.3f}–{d['f1_ci95'][1]:.3f} {d['parse_valid_rate']:6.3f} "
          f"{d['fp_per_image']:7.2f} {pc['pothole']['f1']:8.3f} {pc['crack']['f1']:7.3f} {pc['manhole']['f1']:8.3f} "
          f"{m.get('trainable_params', 0) / 1e6:8.2f}M {m.get('peak_vram_gb', 0):6.2f} {m.get('train_minutes', 0):6.1f}")
bundle = Path('work/bundle'); shutil.rmtree(bundle, ignore_errors=True); (bundle / 'runs').mkdir(parents=True)
shutil.copytree('report/evidence', bundle / 'evidence')
for name in ('E1', 'E2', 'E3'):
    for f in ('run.json', 'train_log.txt'):
        if (RUNS_DIR / name / f).exists():
            shutil.copy(RUNS_DIR / name / f, bundle / 'runs' / f'{name}_{f}')
zip_path = shutil.make_archive(str(DRIVE_DIR / 'bundle_e0_e3') if DRIVE_DIR.exists() else 'work/bundle_e0_e3', 'zip', bundle)
print('\nđã đóng gói bằng chứng:', zip_path, '(tải file zip này về máy để đưa vào repo)')

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9